# Enron Email Spam Detector

## Load the dataset

In [1]:
import pandas as pd 

df=pd.read_csv("enron_spam_data.csv")
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [2]:
df.shape

(33716, 5)

## Explore the Dataset

In [3]:
#check the missing values
print(df.isnull().sum())

Unnamed: 0     0
Subject        0
Message       52
Spam/Ham       0
Date           0
dtype: int64


In [4]:
#remove the missing values
df = df.dropna(subset=['Message'])

print(df.isnull().sum())

Unnamed: 0    0
Subject       0
Message       0
Spam/Ham      0
Date          0
dtype: int64


In [5]:
# Select required columns
df = df[['Subject', 'Message', 'Spam/Ham']]

print(df.head())

                        Subject  \
1      vastar resources , inc .   
2  calpine daily gas nomination   
3                    re : issue   
4     meter 7268 nov allocation   
5      mcmullen gas for 11 / 99   

                                             Message Spam/Ham  
1  gary , production from the high island larger ...      ham  
2             - calpine daily gas nomination 1 . doc      ham  
3  fyi - see note below - already done .\nstella\...      ham  
4  fyi .\n- - - - - - - - - - - - - - - - - - - -...      ham  
5  jackie ,\nsince the inlet to 3 river plant is ...      ham  


## Combine Subject and Message

In [6]:
df['text'] = df['Subject']+" "+df['Message']

print(df[['text', 'Spam/Ham']].head())

                                                text Spam/Ham
1  vastar resources , inc . gary , production fro...      ham
2  calpine daily gas nomination - calpine daily g...      ham
3  re : issue fyi - see note below - already done...      ham
4  meter 7268 nov allocation fyi .\n- - - - - - -...      ham
5  mcmullen gas for 11 / 99 jackie ,\nsince the i...      ham


## Text Preprocessing

In [7]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# First time only
# nltk.download('punkt')
# nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess(text):
    
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Tokenization
    words = word_tokenize(text)
    
    # 4. Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess)

print(df[['text', 'clean_text']].head())

                                                text  \
1  vastar resources , inc . gary , production fro...   
2  calpine daily gas nomination - calpine daily g...   
3  re : issue fyi - see note below - already done...   
4  meter 7268 nov allocation fyi .\n- - - - - - -...   
5  mcmullen gas for 11 / 99 jackie ,\nsince the i...   

                                          clean_text  
1  vastar resources inc gary production high isla...  
2  calpine daily gas nomination calpine daily gas...  
3  issue fyi see note already done stella forward...  
4  meter nov allocation fyi forwarded lauri allen...  
5  mcmullen gas jackie since inlet river plant sh...  


## Convert Text into Numeric Features

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['clean_text'])

print(X.shape)

(33664, 50568)


## Label Encoding

In [9]:
df['Spam/Ham'] = df['Spam/Ham'].map({
    'ham': 0,
    'spam': 1
})

print(df['Spam/Ham'].head())

1    0
2    0
3    0
4    0
5    0
Name: Spam/Ham, dtype: int64


## Train-Test Split

In [10]:
from sklearn.model_selection import train_test_split

y = df['Spam/Ham']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(26931, 50568)
(6733, 50568)


## Model Training (Naive Bayes)

In [11]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train);

## Prediction

In [12]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[1 1 0 0 1 0 0 0 1 0]


## Model Evaluation

In [13]:
from sklearn.metrics import accuracy_score, precision_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("F1 Score:", f1)

Accuracy: 0.9809891578791029
Precision: 0.9643652561247216
F1 Score: 0.981859410430839


## Model Training (Logistic Regression)

In [14]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train);

## Prediction

In [15]:
y_pred_lr = lr_model.predict(X_test)

print(y_pred_lr[:10])

[1 1 0 0 1 0 0 0 1 0]


## Model Evaluation

In [16]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("Accuracy:", accuracy_lr)
print("Precision:", precision_lr)
print("F1 Score:", f1_lr)

Accuracy: 0.9989603445715134
Precision: 0.9979832901181216
F1 Score: 0.9989906272530642


## Model Comparison

In [17]:
comparison = {
    "Model": ["Naive Bayes", "Logistic Regression"],
    "Accuracy": [accuracy, accuracy_lr],
    "Precision": [precision, precision_lr],
    "F1 Score": [f1, f1_lr]
}

comparison_df = pd.DataFrame(comparison)

print(comparison_df.round(4))

                 Model  Accuracy  Precision  F1 Score
0          Naive Bayes     0.981     0.9644    0.9819
1  Logistic Regression     0.999     0.9980    0.9990


## Custom Email Prediction

In [18]:
email = ["Congratulations! You have won a free iPhone. Click the link below to claim your prize."]

email_text = preprocess(email[0])

email_tfidf = tfidf.transform([email_text])

nb_prediction = model.predict(email_tfidf)
lr_prediction = lr_model.predict(email_tfidf)

print("Naive Bayes Prediction:", "Spam" if nb_prediction[0] == 1 else "Ham")
print("Logistic Regression Prediction:", "Spam" if lr_prediction[0] == 1 else "Ham")

Naive Bayes Prediction: Ham
Logistic Regression Prediction: Ham


## Conclusion

In [19]:
print("Naive Bayes Accuracy:", accuracy)
print("Logistic Regression Accuracy:", accuracy_lr)

if accuracy > accuracy_lr:
    print("Naive Bayes performed better on this dataset.")
else:
    print("Logistic Regression performed better on this dataset.")

Naive Bayes Accuracy: 0.9809891578791029
Logistic Regression Accuracy: 0.9989603445715134
Logistic Regression performed better on this dataset.
